In [ ]:
from netgen.meshing import Mesh
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

import numpy as np

In [ ]:
def CapacitorGeometry():

    air = MoveTo(0, 0).RectangleC(30, 30).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    electrode_positive = MoveTo(0, 1).RectangleC(5, 0.5).Face()
    electrode_positive.edges.name = "electrode_positive"
    electrode_positive.faces.name = "electrode_positive"

    electrode_negative = MoveTo(0, -1).RectangleC(5, 0.5).Face()
    electrode_negative.edges.name = "electrode_negative"
    electrode_negative.faces.name = "electrode_negative"

    dielectric = MoveTo(0, 0).RectangleC(4, 1.5).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - electrode_positive - electrode_negative
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, epsr):

    fes_potential = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes_potential.TrialFunction()
    v = fes_potential.TestFunction()

    solution_gf = GridFunction(fes_potential)
    solution_gf.Interpolate(mesh.BoundaryCF({"electrode_positive":1, "electrode_negative":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes_potential.FreeDofs())
    solution_gf.vec.data -= inv@a.mat * solution_gf.vec

    return solution_gf


def CapacitorErrorEstimator(mesh, phi_gf, epsr):

    fes_flux = HDiv(mesh, order=FE_order-1)

    E_gf = GridFunction(fes_flux)
    E = -epsr*grad(phi_gf)

    E_gf.Set(E)

    error_gf = 1/epsr*(E - E_gf)*(E - E_gf)
    error_ZZ = Integrate(error_gf, mesh, VOL, element_wise=True)

    return error_gf, error_ZZ


In [ ]:
h_max = 2
geo = CapacitorGeometry()
mesh = CapacitorMesh(geo, h_max)

epsr_air, epsr_dielectric = 1.0, 2.0
epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

FE_order = 3

phi_gf = CapacitorSolver(mesh, FE_order, epsr)

In [ ]:
error_gf, error_ZZ = CapacitorErrorEstimator(mesh, phi_gf, epsr)
Draw(error_gf, mesh, min=0, max=0.1)

In [ ]:
print(np.array(error_ZZ)[:10], "...")

In [ ]:
maxerr = max(error_ZZ)
print ("maxerr = ", maxerr)

In [ ]:
for el in mesh.Elements():
    mesh.SetRefinementFlag(el, error_ZZ[el.nr] > 0.25*maxerr)

In [ ]:
mesh.Refine()
phi_gf = CapacitorSolver(mesh, FE_order, epsr)
error_gf, error_ZZ = CapacitorErrorEstimator(mesh, phi_gf, epsr)
maxerr = max(error_ZZ)
print ("maxerr = ", maxerr)

In [ ]:
Draw(error_gf, mesh, min=0, max=0.1)

In [ ]:
mesh.Refine()
gf_phi = CapacitorSolver(mesh, FE_order, epsr)
error_gf, error_ZZ = CapacitorErrorEstimator(mesh, gf_phi, epsr)
maxerr = max(error_ZZ)
print ("maxerr = ", maxerr)

In [ ]:
Draw(error_gf, mesh, min=0, max=0.1)